### Import Libraries

In [35]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.metrics import mean_squared_error, r2_score,classification_report, confusion_matrix
from sklearn.linear_model import LinearRegression,LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR

### Load Cleaned Data

In [26]:
df = pd.read_csv("cleaned_data.csv")

In [27]:
df.head(2)

,study_hours,class_attendance,sleep_hours,exam_score,gender,course,internet_access,sleep_quality,study_method,facility_rating,exam_difficulty
0,2.78,92.9,7.4,58.9,1,6,1,2,0,1,1
1,3.37,64.8,4.6,54.8,1,5,1,0,3,2,2


### Split Features / Target

In [28]:
X=df.drop("exam_score",axis=1)
y=df['exam_score']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.25,random_state=42)

### Define Models

In [29]:
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42),
    "SVR": SVR()
}

### Train & Evaluate All Models

In [30]:
results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    mse = mean_squared_error(y_test, predictions)
    r2 = r2_score(y_test, predictions)

    results[name] = {"MSE": mse, "R2": r2}

### Show Results

In [31]:
for model, score in results.items():
    print(f"\n🔹 {model}")
    print(f"   MSE: {score['MSE']:.4f}")
    print(f"   R² : {score['R2']:.4f}")


🔹 Linear Regression
   MSE: 119.1412
   R² : 0.6699

🔹 Decision Tree
   MSE: 224.3299
   R² : 0.3784

🔹 Random Forest
   MSE: 109.8442
   R² : 0.6957

🔹 SVR
   MSE: 147.8185
   R² : 0.5904


In [32]:
best_model = max(results, key=lambda x: results[x]["R2"])
print(f"\n🏆 Best model is: {best_model}")


🏆 Best model is: Random Forest


In [36]:
results = {}

for name, model in models.items():
    # 5-fold cross-validation R²
    cv_scores = cross_val_score(model, X, y, cv=5, scoring='r2')
    
    results[name] = {
        "R2 Mean": np.mean(cv_scores),
        "R2 Std": np.std(cv_scores)
    }

In [37]:
cv_scores

array([0.59433015, 0.59783733, 0.60239865, 0.59995946, 0.60502368])

In [38]:
for model, score in results.items():
    print(f"\n🔹 {model}")
    print(f"   CV R² Mean: {score['R2 Mean']:.4f}")
    print(f"   CV R² Std : {score['R2 Std']:.4f}")



🔹 Linear Regression
   CV R² Mean: 0.6687
   CV R² Std : 0.0077

🔹 Decision Tree
   CV R² Mean: 0.3925
   CV R² Std : 0.0139

🔹 Random Forest
   CV R² Mean: 0.6980
   CV R² Std : 0.0058

🔹 SVR
   CV R² Mean: 0.5999
   CV R² Std : 0.0037


In [39]:
best_model_name = max(results, key=lambda x: results[x]["R2 Mean"])
print(f"\n🏆 Best model based on CV R²: {best_model_name}")



🏆 Best model based on CV R²: Random Forest


In [40]:
best_model = models[best_model_name]
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

print("\nFinal Test Metrics:")
print(f"MSE: {mean_squared_error(y_test, y_pred):.4f}")
print(f"R² : {r2_score(y_test, y_pred):.4f}")



Final Test Metrics:
MSE: 109.8442
R² : 0.6957
